# Reproducibility — inspect a manifest

1. Load a raw JSON manifest from `research/captures/`.
2. Resolve its `runConfig` from whichever source the manifest uses (embedded for v2, sidecar YAML for v1 — same fallback Go ingest applies).
3. Show `manifestVersion` forward-compat semantics.
4. Demo the `self_eval_guard` warning.

Paths resolve via a `go.mod` walk-up so this works from both the tracked location and `.reporting/notebooks/`.

In [ ]:
from pathlib import Path
import json
import os

import yaml  # from pyyaml (analyzer dep)


def _repo_root():
    here = Path(os.getcwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "go.mod").exists():
            return p
    raise SystemExit("repo root (go.mod) not found — run via scripts/report.sh")


ROOT = _repo_root()
CAPTURES_ROOT = ROOT / "research" / "captures"
DB_PATH = ROOT / "reports" / "resolver.duckdb"

In [ ]:
manifest_path = None
if CAPTURES_ROOT.exists():
    for manifests_dir in sorted(CAPTURES_ROOT.glob("*/*/manifests")):
        run_dir = manifests_dir.parent
        if (run_dir / "scorecard.json").exists():
            candidates = sorted(manifests_dir.glob("*.json"))
            if candidates:
                manifest_path = candidates[0]
                break

# Resolve runConfig from whichever source the manifest version uses:
#   v2 manifest  -> embedded in manifest["runConfig"]
#   v1 manifest  -> sibling run-config.yaml in the run directory (same
#                   shape the Go aggregator loads as a sidecar at ingest)
run_config = {}
config_source = "none"

if manifest_path is not None:
    m = json.loads(manifest_path.read_text())
    print(f"using manifest: {manifest_path.relative_to(ROOT)}")
    print(f"  manifestVersion: {m.get('manifestVersion', 1)}")

    if m.get("runConfig"):
        run_config = m["runConfig"]
        config_source = "embedded in manifest (v2)"
    else:
        run_dir = manifest_path.parent.parent
        sidecar = run_dir / "run-config.yaml"
        if sidecar.exists():
            run_config = yaml.safe_load(sidecar.read_text()) or {}
            config_source = f"sidecar {sidecar.relative_to(ROOT)}"
        else:
            config_source = "none (v1 manifest with no sidecar)"
else:
    print("no manifests under research/captures/ — using synthetic fallback.")
    m = {"manifestVersion": 2, "runId": "synthetic-001", "model": "gresh-general", "tier": "1"}
    run_config = {
        "virtual_model": "gresh-general",
        "real_model": "SomeOrg/SomeModel",
        "repeat_group": "group-alpha",
        "notes": "synthetic fallback for notebook demo",
    }
    config_source = "synthetic fallback"

print(f"  runConfig source: {config_source}")

## runConfig and repeat_group

`repeat_group` (snake_case in sidecars, `repeatGroup` in v2 manifests) ties runs from the same `--n N` invocation so they can be compared statistically.

In [ ]:
print(json.dumps(run_config, indent=2, default=str, sort_keys=True))

# Accept either YAML-style (snake_case) or embedded-manifest (camelCase) keys.
repeat_group = run_config.get("repeat_group") or run_config.get("repeatGroup")
real_model = run_config.get("real_model") or run_config.get("realModel")
print(f"\nrepeat_group: {repeat_group!r}")
print(f"real_model:   {real_model!r}")

## manifestVersion forward-compat

- `1` — original schema; `runConfig` lives in a sibling `run-config.yaml` sidecar.
- `2` — current schema; `runConfig` embedded in the manifest.
- `>2` — forward-compat: Go ingest warns and continues, silently ignoring unknown fields.

In [ ]:
CURRENT_SCHEMA_VERSION = 2
manifest_version = m.get("manifestVersion", 1)
print(f"manifestVersion in file: {manifest_version}")
print(f"current SchemaVersion  : {CURRENT_SCHEMA_VERSION}")
if manifest_version > CURRENT_SCHEMA_VERSION:
    print(
        f"WARN: manifestVersion={manifest_version} > {CURRENT_SCHEMA_VERSION}. "
        "Go ingest emits a forward-compat warning for this file."
    )
else:
    print("schema version is current or older — no forward-compat warning.")

## self_eval_guard (demo)

If the reporter LLM is also a model under test, the benchmark becomes self-grading. The cell below deliberately triggers the guard so you can see what the warning looks like — it is meant to appear, not a bug.

In [ ]:
import io
import sys

if not DB_PATH.exists():
    print(f"DuckDB missing at {DB_PATH} — run scripts/report.sh first.")
else:
    from analyze.report import self_eval_guard
    from analyze.db import Store

    with Store(DB_PATH) as s:
        runs = s.run_summaries()

    # Deliberately pass one of the tested models as the reporter so you
    # can see the warning. In real use, `--reporter-model` should be a
    # separate model that was NOT tested in this corpus.
    reporter = runs[0].model if runs else "gresh-general"
    print(f"demo: calling self_eval_guard with reporter_model={reporter!r}")
    print("      (this reporter is in the tested set, so the warning SHOULD fire)")
    print()

    buf = io.StringIO()
    old_stderr, sys.stderr = sys.stderr, buf
    try:
        self_eval_guard(runs, reporter)
    finally:
        sys.stderr = old_stderr

    captured = buf.getvalue()
    if captured:
        print(captured.rstrip())
    else:
        print("no warning emitted (unexpected — reporter should have been in the set).")

## Done

- See **`quickstart.ipynb`** for the dashboard view.
- Re-run `scripts/report.sh` after adding captures.